# 🍄 슈퍼마리오 DQN 학습
**런타임 → 런타임 유형 변경 → T4 GPU 선택 후 실행하세요**

## 1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PATH = '/content/drive/MyDrive/supermario_dl_project'
os.makedirs(f'{DRIVE_PATH}/models', exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/results', exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/videos', exist_ok=True)
print('Drive 마운트 완료!')

## 2. 패키지 설치

In [ ]:
!apt-get install -y xvfb cmake fonts-nanum > /dev/null 2>&1

!pip install --upgrade pip wheel -q
!pip install "setuptools>=65.5.1,<82" -q
!pip install nes-py==8.2.1 --no-build-isolation -q
!pip install gym==0.26.2 -q
!pip install gym-super-mario-bros==7.4.0 --no-deps -q
!pip install opencv-python-headless imageio imageio-ffmpeg pyvirtualdisplay tqdm -q

import glob, re, os, sys, numpy as np

# [한글 폰트] NanumGothic 절대경로로 직접 등록 (Colab 표준 경로)
import matplotlib
import matplotlib.font_manager as fm

_NANUM_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
if os.path.exists(_NANUM_PATH):
    fm.fontManager.addfont(_NANUM_PATH)
    matplotlib.rcParams['font.family'] = fm.FontProperties(fname=_NANUM_PATH).get_name()
    print(f'[폰트] NanumGothic 적용')
else:
    # 폰트 파일이 다른 경로에 있을 경우 탐색
    _found = glob.glob('/usr/share/fonts/**/*anum*.ttf', recursive=True)
    if _found:
        fm.fontManager.addfont(_found[0])
        matplotlib.rcParams['font.family'] = fm.FontProperties(fname=_found[0]).get_name()
        print(f'[폰트] {os.path.basename(_found[0])} 적용')
    else:
        print('[폰트] NanumGothic 미발견 — 한글이 깨질 수 있음')
matplotlib.rcParams['axes.unicode_minus'] = False

# [패치 0] numpy 런타임: np.bool8 직접 등록
if not hasattr(np, 'bool8'):
    np.bool8 = np.bool_
    print('[numpy] np.bool8 등록')

# [패치 1] nes_py, gym_super_mario_bros: numpy 2.0 uint8 오버플로우 종합 수정
files = (
    glob.glob('/usr/local/lib/python*/dist-packages/nes_py/*.py') +
    glob.glob('/usr/local/lib/python*/dist-packages/gym_super_mario_bros/*.py')
)
p1 = re.compile(r'(self\.ram\[[^\]]+\]|self\.\w+_size)\s*\*\s*((?:0x[0-9a-fA-F]+|\d+)(?:\*\*\d+)?)')
p2 = re.compile(r'(?<!int\()self\.ram\[([^\]:]+)\](?!\s*[=\[])')
total = 0
for path in files:
    with open(path) as f:
        content = f.read()
    patched = p1.sub(lambda m: f'int({m.group(1)}) * {m.group(2)}', content)
    patched = p2.sub(r'int(self.ram[\1])', patched)
    if patched != content:
        with open(path, 'w') as f:
            f.write(patched)
        for pyc in glob.glob(os.path.join(os.path.dirname(path), '__pycache__',
                             os.path.basename(path).replace('.py', '*.pyc'))):
            os.remove(pyc)
        print(f'  [uint8] {path.split("/")[-1]}')
        total += 1
print(f'uint8 패치: {total}개 파일')

# [패치 2] passive_env_checker: np.bool8 제거됨
for path in glob.glob('/usr/local/lib/python*/dist-packages/gym/utils/passive_env_checker.py'):
    with open(path) as f:
        content = f.read()
    patched = content.replace('np.bool8', 'np.bool_')
    if patched != content:
        with open(path, 'w') as f:
            f.write(patched)
        for pyc in glob.glob(os.path.join(os.path.dirname(path), '__pycache__', 'passive_env_checker*.pyc')):
            os.remove(pyc)
        print('[bool8] passive_env_checker.py 패치 완료')

# [패치 3] time_limit: old API(4-return) 호환
OLD = 'observation, reward, terminated, truncated, info = self.env.step(action)'
NEW = ('result = self.env.step(action)\n'
       '        if len(result) == 4:\n'
       '            observation, reward, terminated, info = result\n'
       '            truncated = False\n'
       '        else:\n'
       '            observation, reward, terminated, truncated, info = result')
for path in glob.glob('/usr/local/lib/python*/dist-packages/gym/wrappers/time_limit.py'):
    with open(path) as f:
        content = f.read()
    if OLD in content:
        with open(path, 'w') as f:
            f.write(content.replace(OLD, NEW))
        for pyc in glob.glob(os.path.join(os.path.dirname(path), '__pycache__', 'time_limit*.pyc')):
            os.remove(pyc)
        print('[time_limit] 4/5-return 호환 패치 완료')

# 모듈 캐시 초기화
cleared = [k for k in list(sys.modules.keys())
           if k.startswith(('gym', 'nes_py', 'gym_super_mario_bros'))]
for k in cleared:
    del sys.modules[k]
print(f'모듈 캐시 초기화: {len(cleared)}개')
print('설치 완료!')

In [ ]:
# ===== 한글 폰트 테스트 =====
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import glob, os

_NANUM_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
if not os.path.exists(_NANUM_PATH):
    _found = glob.glob('/usr/share/fonts/**/*anum*.ttf', recursive=True)
    _NANUM_PATH = _found[0] if _found else None

if _NANUM_PATH and os.path.exists(_NANUM_PATH):
    fm.fontManager.addfont(_NANUM_PATH)
    prop = fm.FontProperties(fname=_NANUM_PATH)
    font_name = prop.get_name()
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = [font_name] + plt.rcParams.get('font.sans-serif', [])
    print(f'[폰트] {font_name} 적용 성공')
else:
    print('[폰트] NanumGothic 파일 없음 — 한글 깨질 수 있음')
plt.rcParams['axes.unicode_minus'] = False

fig, ax = plt.subplots(figsize=(6, 3))
ax.set_title('한글 폰트 테스트', fontsize=14, fontweight='bold')
ax.text(0.5, 0.6, '에피소드 보상 / 학습 곡선', ha='center', va='center',
        fontsize=13, transform=ax.transAxes)
ax.text(0.5, 0.35, '현재 ε (epsilon): 1.0000', ha='center', va='center',
        fontsize=11, color='gray', transform=ax.transAxes)
ax.axis('off')
plt.tight_layout()
plt.savefig('/content/test_korean_font.png', dpi=120, bbox_inches='tight')
plt.show()
print('저장 완료: /content/test_korean_font.png')
print('위 이미지에서 한글이 정상 출력되면 OK!')

## 3. 가상 디스플레이 시작 (Colab 헤드리스 렌더링)

In [ ]:
from pyvirtualdisplay import Display
display = Display(visible=0, size=(400, 300))
display.start()
print('가상 디스플레이 시작!')

## 4. GitHub에서 프로젝트 클론

In [ ]:
GITHUB_REPO = 'https://github.com/HyunDove/supermario-dqn.git'

import os
if not os.path.exists('/content/supermario-dqn'):
    !git clone {GITHUB_REPO} /content/supermario-dqn
else:
    !cd /content/supermario-dqn && git pull

os.chdir('/content/supermario-dqn')
print('프로젝트 준비 완료!')

## 5. GPU 확인

In [ ]:
import torch
print(f'GPU 사용 가능: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU 모델: {torch.cuda.get_device_name(0)}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'사용 디바이스: {device}')

## 6. TensorBoard 실행 (학습 전에 먼저 실행하세요)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {DRIVE_PATH}/logs

## 7. 학습 실행 (영상 자동 기록 포함)

## 8. 학습 곡선 그래프 저장

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.ticker as ticker
import numpy as np
import glob, os

# 한글 폰트 — 그래프 저장 직전 재적용 (글로벌 rcParams 의존 안 함)
_NANUM_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
if not os.path.exists(_NANUM_PATH):
    _found = glob.glob('/usr/share/fonts/**/*anum*.ttf', recursive=True)
    _NANUM_PATH = _found[0] if _found else None
if _NANUM_PATH and os.path.exists(_NANUM_PATH):
    fm.fontManager.addfont(_NANUM_PATH)
    prop = fm.FontProperties(fname=_NANUM_PATH)
    font_name = prop.get_name()
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = [font_name] + plt.rcParams.get('font.sans-serif', [])
    print(f'[폰트] {font_name} 적용')
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle('슈퍼마리오 DQN 학습 결과', fontsize=14, fontweight='bold')

ax1 = axes[0]
ax1.plot(rewards_history, alpha=0.3, color='steelblue', label='에피소드 보상')
window = min(100, len(rewards_history))
avg = np.convolve(rewards_history, np.ones(window)/window, mode='valid')
ax1.plot(range(window-1, len(rewards_history)), avg, color='tomato', linewidth=2, label=f'{window}회 이동 평균')

record_points = [ep for ep in [0, 1000, 5000, 7000] if ep < len(rewards_history)]
for ep in record_points:
    ax1.axvline(x=ep, color='gold', linestyle='--', alpha=0.7, linewidth=1.2)
    ax1.text(ep, ax1.get_ylim()[1]*0.9, f'EP{ep}\n영상', fontsize=7, ha='center', color='goldenrod')

ax1.set_xlabel('에피소드')
ax1.set_ylabel('총 보상')
ax1.set_title('에피소드별 보상 변화')
ax1.legend()
ax1.grid(alpha=0.3)

ax2 = axes[1]
ax2.plot(range(window-1, len(rewards_history)), avg, color='tomato', linewidth=2)
ax2.fill_between(range(window-1, len(rewards_history)), avg, alpha=0.2, color='tomato')
ax2.set_xlabel('에피소드')
ax2.set_ylabel('평균 보상 (100회)')
ax2.set_title('학습 안정화 추세 (100회 이동 평균)')
ax2.grid(alpha=0.3)

plt.tight_layout()
save_path = f'{DRIVE_PATH}/results/training_curve.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'그래프 저장: {save_path}')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np
import glob, os

_NANUM_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
if not os.path.exists(_NANUM_PATH):
    _found = glob.glob('/usr/share/fonts/**/*anum*.ttf', recursive=True)
    _NANUM_PATH = _found[0] if _found else None
if _NANUM_PATH and os.path.exists(_NANUM_PATH):
    fm.fontManager.addfont(_NANUM_PATH)
    prop = fm.FontProperties(fname=_NANUM_PATH)
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = [prop.get_name()] + plt.rcParams.get('font.sans-serif', [])
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle('슈퍼마리오 DQN 학습 결과', fontsize=14, fontweight='bold')

ax1 = axes[0]
ax1.plot(rewards_history, alpha=0.3, color='steelblue', label='에피소드 보상')
window = min(100, len(rewards_history))
avg = np.convolve(rewards_history, np.ones(window)/window, mode='valid')
ax1.plot(range(window-1, len(rewards_history)), avg, color='tomato', linewidth=2, label=f'{window}회 이동 평균')
for ep in [ep for ep in [0, 1000, 5000, 10000] if ep < len(rewards_history)]:
    ax1.axvline(x=ep, color='gold', linestyle='--', alpha=0.7, linewidth=1.2)
    ax1.text(ep, ax1.get_ylim()[1]*0.9, f'EP{ep}\n영상', fontsize=7, ha='center', color='goldenrod')
ax1.set_xlabel('에피소드')
ax1.set_ylabel('총 보상')
ax1.set_title('에피소드별 보상 변화')
ax1.legend()
ax1.grid(alpha=0.3)

ax2 = axes[1]
ax2.plot(range(window-1, len(rewards_history)), avg, color='tomato', linewidth=2)
ax2.fill_between(range(window-1, len(rewards_history)), avg, alpha=0.2, color='tomato')
ax2.set_xlabel('에피소드')
ax2.set_ylabel('평균 보상 (100회)')
ax2.set_title('학습 안정화 추세 (100회 이동 평균)')
ax2.grid(alpha=0.3)

plt.tight_layout()
save_path = f'{DRIVE_PATH}/results/training_curve.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'그래프 저장: {save_path}')

## 9. 저장된 파일 목록 확인

## 10. 세션 끊긴 후 이어서 학습하기

## 9. 세션 끊긴 후 이어서 학습하기

In [ ]:
# 1~5번 셀 재실행 후, 6번 셀에서 CHECKPOINT_PATH 수정
# 예시:
# CHECKPOINT_PATH = f'{DRIVE_PATH}/models/mario_ep1000.pth'

import os
print('마지막 저장 모델:')
models = sorted(os.listdir(f'{DRIVE_PATH}/models'))
for m in models[-3:]:
    print(f'  {DRIVE_PATH}/models/{m}')